You are given an array prices where prices[i] is the price of a given stock on the ith day.

Find the maximum profit you can achieve. You may complete as many transactions as you like (i.e., buy one and sell one share of the stock multiple times) with the following restrictions:

After you sell your stock, you cannot buy stock on the next day (i.e., cooldown one day).
Note: You may not engage in multiple transactions simultaneously (i.e., you must sell the stock before you buy again).

 

Example 1:

Input: prices = [1,2,3,0,2]
Output: 3
Explanation: transactions = [buy, sell, cooldown, buy, sell]
Example 2:

Input: prices = [1]
Output: 0
 

Constraints:

1 <= prices.length <= 5000
0 <= prices[i] <= 1000

In [ ]:
class Solution:
    def maxProfit(self, prices: list[int]) -> int:
        
        def dsf(index, buy):
            if index == len(prices):
                return 0

            if buy:
                buy_now = -prices[index] + dsf(index + 1, False) # buy now can't buy next
                not_buy_now = 0 + dsf(index + 1, True)
                return max(buy_now, not_buy_now)
            else:
                sell_now = prices[index] + dsf(index + 2, True)
                not_sell_now = 0 + dsf(index + 1, False)
                return max(sell_now, not_sell_now)
            
        return dsf(0, True)

# tc - O(2^n) 
# sc - O(n) for recursion stack

In [8]:
# memoization:
class Solution:
    def maxProfit(self, prices: list[int]) -> int:
        dp = [[-1]* 2 for _ in range(len(prices))]

        def dsf(index, buy):
            if index >= len(prices):
                return 0

            if dp[index][buy] != -1:
                return dp[index][buy]
            
            if buy:
                buy_now = -prices[index] + dsf(index + 1, False) # buy now can't buy next
                not_buy_now = 0 + dsf(index + 1, True)
                dp[index][buy] =  max(buy_now, not_buy_now)
            else:
                sell_now = prices[index] + dsf(index + 2, True)
                not_sell_now = 0 + dsf(index + 1, False)
                dp[index][buy] = max(sell_now, not_sell_now)
            
            return dp[index][buy]
            
        return dsf(0, True)

# tc - O(n * 2)
# sc - O(n)

In [9]:
Solution().maxProfit(prices = [1,2,3,0,2])

3

In [ ]:
# tabulation:
class Solution:
    def maxProfit(self, prices: list[int]) -> int:
        n = len(prices)
        # it will look at n when i == n - 2:
        # so make the array upto n.
        dp = [[0]* 2 for _ in range(len(prices) + 2)]

        # base case:
        # when the last day, you can sell the current one if possible.
        dp[n-1][True] = 0 # we can buy now, but no use of buying n the last day.
        dp[n-1][False] = prices[n-1] # we can sell now, so profit is prices[n-1].

        for index in range(n-2, -1, -1):
            for buy in range(0, 2):
                print(index, buy)
                if buy:
                    buy_now = -prices[index] + dp[index + 1][False] # buy now can't buy next
                    not_buy_now = 0 + dp[index + 1][True]
                    dp[index][buy] =  max(buy_now, not_buy_now)
                
                else:
                    sell_now = prices[index] + dp[index + 2][True]
                    not_sell_now = 0 + dp[index + 1][False]
                    dp[index][buy] = max(sell_now, not_sell_now)
        return dp[0][True] # # we can buy at the start, so return dp[0][True] NOTE

# tc- O(n * 2)
# sc - O(n * 2)

In [26]:
Solution().maxProfit(prices = [1,2,3,0,2])

3 0
3 1
2 0
2 1
1 0
1 1
0 0
0 1


3

In [ ]:
class Solution:
    def maxProfit(self, prices: list[int], cooldown: int) -> int:

        def dfs(ind: int, trans:int ) -> int:
            if ind >= len(prices):
                return 0
            if ind == len(prices) - 1 :
                if trans % 2 != 0:
                    return prices[ind]
                return 0
            max_buy = float("-inf")
            max_sell = float("-inf")
            if trans% 2 == 0:
                # buy it
                buy = dfs(ind + 1, trans + 1) - prices[ind]
                not_buy = dfs(ind + 1, trans)
                max_buy = max(buy, not_buy)
            else:
                # sell state
                # Option 1: sell today
                if ind + 1 + cooldown < n:
                    sell = dfs(ind + 1 + cooldown, trans + 1) + prices[ind]
                else:
                    # "If I sell today at ind, my cooldown period extends to day ind + cooldown,
                    # which is past the end of the array — so I can never buy again."
                    sell = prices[ind]
                not_sell = dfs(ind + 1, trans)
                max_sell = max(sell, not_sell)
            
            return max(max_buy, max_sell)

        return dfs(0, 0)


In [7]:
Solution().maxProfit(prices = [1,2,3,0,2], cooldown=1)

3

In [ ]:
# tabulation
class Solution:
    def maxProfit(self, prices: list[int], k: int, cooldown: int) -> int:
        
        num_of_tran = 2*k
        n = len(prices)

        if n<=1 or k == 0:
            return 0
        
        dp = [[0] * (2*k+1) for _ in range(len(prices))]
        # set the base condition:
        for i in range(num_of_tran):
            if i % 2 != 0:
                dp[n-1][i] = prices[-1]

        for ind in range(n-2, -1, -1):
            for trans in range(num_of_tran):

                if trans % 2 == 0:
                    # buy it
                    buy = dp[ind + 1][trans + 1] - prices[ind]
                    not_buy = dp[ind + 1][trans]
                    dp[ind][trans] = max(buy, not_buy)
                else:
                    sell = 0
                    # sell state
                    # Option 1: sell today
                    if ind + 1 + cooldown < n:
                        sell = dp[ind + 1 + cooldown][trans + 1] + prices[ind]
                    else:
                        # "If I sell today at ind, my cooldown period extends to day ind + cooldown,
                        # which is past the end of the array — so I can never buy again."
                        sell = prices[ind] # but no futhur transactions
                        pass
                        
                    not_sell = dp[ind + 1][trans]
                    dp[ind][trans] = max(sell, not_sell)
        print(dp)
        return dp[0][0]


In [31]:
Solution().maxProfit(prices = [1,2,3,0,2], k = 2, cooldown=1)

[[3, 4, 2, 3, 0], [2, 4, 2, 3, 0], [2, 3, 2, 3, 0], [2, 2, 2, 2, 0], [0, 2, 0, 2, 0]]


3

In [32]:
Solution().maxProfit(prices = [1,5,3,6,4], k=2, cooldown=1)

[[4, 5, 4, 5, 0], [1, 5, 1, 5, 0], [1, 4, 1, 4, 0], [0, 4, 0, 4, 0], [0, 4, 0, 4, 0]]


4

In [33]:
Solution().maxProfit([1,2,3,0,2], k=2, cooldown=1)  # should return 3
Solution().maxProfit([1,2,3], k=1, cooldown=1)       # should return 2
Solution().maxProfit([2,1], k=1, cooldown=1)         # should return 0


[[3, 4, 2, 3, 0], [2, 4, 2, 3, 0], [2, 3, 2, 3, 0], [2, 2, 2, 2, 0], [0, 2, 0, 2, 0]]
[[2, 3, 0], [1, 3, 0], [0, 3, 0]]
[[0, 1, 0], [0, 1, 0]]


0